# Exercise 2: video → frames → JSON timeline

## Tasks

1. **Frames:** take one frame per second from Ueli's 8-second clip.
2. **Describe:** ask the model to describe each frame as JSON.
3. **Transcribe:** turn the speech into text with start and end times.
4. **Merge:** join frames and speech on time and detect the events.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
import base64
import json
import mimetypes
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openai import OpenAI
from PIL import Image
from pydantic import BaseModel

# Work inside this exercise folder, even if Jupyter starts in the main folder.
if Path("data").exists():
    os.chdir("02_Video_Timeline")

client = OpenAI(max_retries=5, timeout=60)  # reads OPENAI_API_KEY, retries if the API is busy
MODEL = "gpt-6-luna"

DATA = Path("..") / "data"  # the course images and videos

## Helper functions

Two small functions do the API work in this notebook:

- `image_to_input(path)` turns an image file into text (base64), because the API request is sent as text.
- `ask(prompt, answer_format, images)` sends the prompt and the images, writes a log line and returns the answer as a Python dict.

In [ ]:
def image_to_input(path):
    """Turn an image file into the text format the API expects."""
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


def ask(prompt, answer_format, images):
    """Send a prompt and a list of images. Returns the answer as a dict."""
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Stop with a clear message if there is no valid answer.
    if response.output_parsed is None:
        raise ValueError(f"No valid answer for {images}. Look at api_log.jsonl.")

    # 5. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

## 1. Frames

One frame per second with OpenCV. The frames are saved in `frames/` – you can open them in the Explorer.

In [ ]:
video_path = DATA / "videos" / "ueli_scene3_talk.mp4"

Path("frames").mkdir(exist_ok=True)
video = cv2.VideoCapture(str(video_path))

paths = []
for t in range(8):  # Flow clips are 8 seconds long
    video.set(cv2.CAP_PROP_POS_MSEC, t * 1000)  # jump to second t
    ok, frame = video.read()
    if not ok:  # the video ended earlier
        break
    path = Path("frames") / f"frame_{t}s.jpg"
    cv2.imwrite(str(path), frame)
    paths.append(path)

video.release()
print(len(paths), "frames saved in frames/")

In [ ]:
fig, axes = plt.subplots(1, len(paths), figsize=(2 * len(paths), 2))
for t in range(len(paths)):
    axes[t].imshow(Image.open(paths[t]))
    axes[t].set_title(f"{t} s")
    axes[t].axis("off")
plt.show()

## 2. Describe

While the cells run: create your Google account for Flow.

In [ ]:
class Frame(BaseModel):
    description: str
    faces_camera: bool


PROMPT = "Describe this video frame in one short sentence. Is Ueli looking toward the camera?"

In [ ]:
rows = []
for t in range(len(paths)):
    row = ask(PROMPT, Frame, [paths[t]])
    row["t"] = t  # we know the time, the model does not need to
    rows.append(row)

frames = pd.DataFrame(rows)
frames

## 3. Transcribe

Speech-to-text with start and end time per segment. Swiss German often comes back as Standard German.

In [ ]:
with open(video_path, "rb") as audio:  # Whisper reads the sound from the mp4
    response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio,
        response_format="verbose_json",
        timestamp_granularities=["segment"],
    )

entry = {"prompt": "transcribe", "model": "whisper-1", "answer": response.text}
with open("api_log.jsonl", "a", encoding="utf-8") as log:
    log.write(json.dumps(entry, ensure_ascii=False) + "\n")

speech = pd.DataFrame(response.model_dump()["segments"])[["start", "end", "text"]]
speech

## 4. Merge on time

A frame belongs to a speech segment when `start ≤ t < end`.

In [ ]:
frames["said"] = ""
for i, segment in speech.iterrows():
    inside = frames["t"].between(segment["start"], segment["end"], inclusive="left")
    frames.loc[inside, "said"] = segment["text"]

frames

Is every spoken line in the table? A short line between two frames is missed – that is why speech is also kept as its own event below.

## 5. Events

Collapse the frames into events: when does Ueli start and stop facing the camera?

In [ ]:
# Number each run of equal values: False False True True True False -> 1 1 2 2 2 3
run = (frames["faces_camera"] != frames["faces_camera"].shift()).cumsum()

# For each run of True: first and last second.
facing = frames[frames["faces_camera"]]
events = facing.groupby(run)["t"].agg(start="min", end="max").reset_index(drop=True)
events["end"] = events["end"] + 1  # the event lasts until the next frame
events["event"] = "faces_camera"
events

Compare with the hand-labelled reference `[2.0, 5.0)`. A match needs both start and end within ±1 s.

In [ ]:
events["match"] = np.isclose(events["start"], 2, atol=1) & np.isclose(events["end"], 5, atol=1)
events

## 6. Timeline

Both kinds of events, sorted by time, saved as JSON lines.

In [ ]:
timeline = pd.concat([events, speech.assign(event="speech")]).sort_values("start")
timeline.to_json("timeline.jsonl", orient="records", lines=True, force_ascii=False)
timeline

**Check yourself:** What would you lose with one frame every 2 seconds instead of every second?

### Jupyter notebook --footer info--

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')